# OffensEval — Classical ML Baselines

TF-IDF + Logistic Regression / Random Forest baselines for the harmful vs. not-harmful
classification task. This is the classical ML half of the project — see
`02_deep_learning.ipynb` for the BiLSTM / CNN+LSTM comparison.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../src')

## 1. Load and explore the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

file_path = "../data/TBO_4k_train.xlsx"
df = pd.read_excel(file_path)

print("Preview of dataset:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values per Column:")
print(df.isnull().sum())
print("\nUnique values in T1 Harmful column:")
print(df['T1 Harmful'].unique())

## 2. Clean and prepare the labels

In [ ]:
df = df[['id', 'text', 'T1 Harmful']]
df['label'] = df['T1 Harmful'].map({'YES': 1, 'NO': 0})
df = df.dropna(subset=['text', 'label'])
df = df.drop_duplicates(subset='text')

print("Cleaned dataset shape:", df.shape)
print("\nClass distribution after cleaning:")
print(df['label'].value_counts())

sns.countplot(x='label', data=df)
plt.title('Class Distribution (0 = Not Harmful, 1 = Harmful)')
plt.show()

## 3. Text cleaning + train/test split

In [ ]:
from preprocessing import clean_text
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

df['clean_text'] = df['text'].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("\nLabel distribution in training set:")
print(y_train.value_counts(normalize=True))

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print("\nComputed Class Weights:", dict(enumerate(class_weights)))

## 4. TF-IDF vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF feature matrix shape (train):", X_train_tfidf.shape)

## 5. Logistic Regression baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from evaluate import evaluate_model

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train_tfidf, y_train)
y_pred = log_reg.predict(X_test_tfidf)

evaluate_model(y_test, y_pred, "Logistic Regression", save_dir="../results")

## 6. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, max_depth=None, n_jobs=-1
)
rf_model.fit(X_train_tfidf, y_train)
y_pred_rf = rf_model.predict(X_test_tfidf)

evaluate_model(y_test, y_pred_rf, "Random Forest", save_dir="../results")

## 7. Random Forest tuned with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid, cv=5, n_jobs=-1, verbose=1
)
grid_rf.fit(X_train_tfidf, y_train)

best_rf = grid_rf.best_estimator_
print("Best params:", grid_rf.best_params_)

y_pred_best_rf = best_rf.predict(X_test_tfidf)
evaluate_model(y_test, y_pred_best_rf, "Random Forest Tuned", save_dir="../results")

## 8. Save the model for the demo app

The Logistic Regression model + TF-IDF vectorizer are saved so `app/app.py` can load
them for live predictions.

In [ ]:
import joblib
import os

os.makedirs("../saved_models", exist_ok=True)
joblib.dump(log_reg, "../saved_models/logreg_model.joblib")
joblib.dump(tfidf, "../saved_models/tfidf_vectorizer.joblib")
print("Saved.")

## Summary

| Model | Accuracy | Macro F1 |
|---|---|---|
| Logistic Regression | 0.709 | 0.69 |
| Random Forest | 0.723 | 0.69 |
| Random Forest (tuned) | 0.711 | 0.69 |

Interestingly, hyperparameter tuning did not improve on the untuned Random Forest —
a reminder that more tuning isn't always better, especially on a small, noisy dataset
like this one. See `02_deep_learning.ipynb` for the BiLSTM / CNN+LSTM comparison, which
outperforms all three of these baselines.